# Bölüm 3 — BÖLÜM 3: Veri Ön İşleme ve Temizleme Teknikleri

**VERİ MADENCİLİĞİ VE MAKİNE ÖĞRENMESİ**  
*Python ile Temel Analitikten Büyük Veri ve Gerçek Zamanlı Sistemlere*

Bu defter, kitabın 3. bölümündeki tüm kod örneklerini içerir. Her hücrenin başlığı kitaptaki alt bölüme karşılık gelir.


In [ ]:
# Bu bölüm için gerekli paketler
!pip install -q matplotlib missingno numpy pandas scikit-learn scipy seaborn torch


## 3.1. Eksik ve Bozuk Verilerin Tespiti


### 3.1.3.1. Programatik Denetim ile Eksik Veri Analizi

`bolum03/03_01_03_01_programatik-denetim-ile-eksik-veri-analizi.py`

_Kitap: Kod 3.1_


In [ ]:
import random
# ─── Eksik veri analizi: temel pandas araçları ───────────────────
import pandas as pd
import numpy as np

np.random.seed(42)
n = 200
df = pd.DataFrame({
    "yas":         np.where(np.random.rand(n) < 0.05, np.nan,
                            np.random.randint(18, 65, n).astype(float)),
    "gelir":       np.where(np.random.rand(n) < 0.12, np.nan,
                            np.random.normal(5000, 1500, n)),
    "egitim_yil":  np.where(np.random.rand(n) < 0.08, np.nan,
                            np.random.randint(8, 22, n).astype(float)),
    "kredi_skoru": np.where(np.random.rand(n) < 0.20, np.nan,
                            np.random.randint(300, 850, n).astype(float)),
    "sehir":       np.random.choice(["Ankara","Istanbul","Izmir",None],
                                     n, p=[0.35,0.35,0.20,0.10]),
})

# Sutun bazında eksiklik ozeti
eksik_df = pd.DataFrame({
    "Eksik Sayisi":   df.isnull().sum(),
    "Eksik Oran (%)": df.isnull().mean().mul(100).round(2),
    "Veri Tipi":      df.dtypes
}).sort_values("Eksik Oran (%)", ascending=False)

print("=== Sutun Bazli Eksiklik Ozeti ===")
print(eksik_df)

# Satir bazinda kac sutunda eksiklik var?
satirbazli = df.isnull().sum(axis=1)
print("\n=== Satir Bazli Eksik Sutun Sayisi Dagilimi ===")
print(satirbazli.value_counts().sort_index())

# Tam eksiksiz satir orani
tam = df.dropna().shape[0]
print(f"\nTam eksiksiz satir: {tam} ({tam/n*100:.1f}%)")


### 3.1.3.2. İstatistiksel Testler ile Mekanizma Tespiti

`bolum03/03_01_03_02_istatistiksel-testler-ile-mekanizma-tespiti.py`

_Kitap: Kod 3.2_


In [ ]:
# ─── Ön hazırlık ─────────────────────────────────────────────────────
# Bu kesim, kitapta bir önceki kesimde kurulan veriyi/modeli kullanır.
# Dosyanın tek başına çalışabilmesi için o hazırlık burada yinelenmiştir.
# Kaynak: bolum03/03_01_03_01_programatik-denetim-ile-eksik-veri-analizi.py
import numpy as np, pandas as pd
np.random.seed(42)
n = 200
df = pd.DataFrame({
    "yas":         np.where(np.random.rand(n) < 0.05, np.nan,
                            np.random.randint(18, 65, n).astype(float)),
    "gelir":       np.where(np.random.rand(n) < 0.12, np.nan,
                            np.random.normal(5000, 1500, n)),
    "egitim_yil":  np.where(np.random.rand(n) < 0.08, np.nan,
                            np.random.randint(8, 22, n).astype(float)),
    "kredi_skoru": np.where(np.random.rand(n) < 0.20, np.nan,
                            np.random.randint(300, 850, n).astype(float)),
    "sehir":       np.random.choice(["Ankara","Istanbul","Izmir",None],
                                     n, p=[0.35,0.35,0.20,0.10]),
})
# ─── Ön hazırlık sonu ────────────────────────────────────────────────

import pandas as pd
# ─── MAR/MCAR hizli kontrolu: t-testi ile gruplar arasi fark ──────
from scipy import stats

# Gelir eksik olan ve olmayan gruplarda "yas" ortalamasini karsilastir
gelir_eksik  = df[df["gelir"].isnull()]["yas"].dropna()
gelir_mevcut = df[df["gelir"].notna()]["yas"].dropna()

t_stat, p_val = stats.ttest_ind(gelir_eksik, gelir_mevcut)

print(f"Gelir eksik grubu   - Yas ort.: {gelir_eksik.mean():.2f} (n={len(gelir_eksik)})")
print(f"Gelir mevcut grubu  - Yas ort.: {gelir_mevcut.mean():.2f} (n={len(gelir_mevcut)})")
print(f"t-istatistigi: {t_stat:.4f},  p-degeri: {p_val:.4f}")

if p_val < 0.05:
    print("=> Anlamli fark var  -> MAR veya MNAR olabilir")
else:
    print("=> Anlamli fark yok  -> MCAR varsayimina yaklasilir")

# Eksiklik gostergesi korelasyon matrisi
gosterge = df.isnull().astype(int)
gosterge.columns = [f"{c}_eksik" for c in gosterge.columns]
corr = pd.concat([df.select_dtypes("number"), gosterge], axis=1).corr()
print("\nEksiklik gosterge korelasyonlari:")
print(corr.loc[df.select_dtypes("number").columns, gosterge.columns].round(3))


### 3.1.3.3. Görselleştirme Teknikleri

`bolum03/03_01_03_03_gorsellestirme-teknikleri.py`

_Kitap: Kod 3.3_


In [ ]:
# ─── Ön hazırlık ─────────────────────────────────────────────────────
# Bu kesim, kitapta bir önceki kesimde kurulan veriyi/modeli kullanır.
# Dosyanın tek başına çalışabilmesi için o hazırlık burada yinelenmiştir.
# Kaynak: bolum03/03_01_03_01_programatik-denetim-ile-eksik-veri-analizi.py
import numpy as np, pandas as pd
np.random.seed(42)
n = 200
df = pd.DataFrame({
    "yas":         np.where(np.random.rand(n) < 0.05, np.nan,
                            np.random.randint(18, 65, n).astype(float)),
    "gelir":       np.where(np.random.rand(n) < 0.12, np.nan,
                            np.random.normal(5000, 1500, n)),
    "egitim_yil":  np.where(np.random.rand(n) < 0.08, np.nan,
                            np.random.randint(8, 22, n).astype(float)),
    "kredi_skoru": np.where(np.random.rand(n) < 0.20, np.nan,
                            np.random.randint(300, 850, n).astype(float)),
    "sehir":       np.random.choice(["Ankara","Istanbul","Izmir",None],
                                     n, p=[0.35,0.35,0.20,0.10]),
})
# ─── Ön hazırlık sonu ────────────────────────────────────────────────

import missingno as msno
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

msno.matrix(df, ax=axes[0,0], sparkline=False, color=(0.18,0.46,0.71))
axes[0,0].set_title("Eksik Veri Matrisi", fontsize=12, fontweight="bold")

msno.bar(df, ax=axes[0,1], color=(0.18,0.46,0.71))
axes[0,1].set_title("Sutun Bazinda Doluluk Orani", fontsize=12, fontweight="bold")

msno.heatmap(df, ax=axes[1,0])
axes[1,0].set_title("Eksiklik Korelasyon Isi Haritasi", fontsize=12, fontweight="bold")

msno.dendrogram(df, ax=axes[1,1])
axes[1,1].set_title("Eksiklik Dendrogrami", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig("eksik_veri_gorsellestirme.png", dpi=150, bbox_inches="tight")
plt.show()


### 3.1.3.4. Eksik Veri Örüntüsü Analizi

`bolum03/03_01_03_04_eksik-veri-oruntusu-analizi.py`

_Kitap: Kod 3.4_


In [ ]:
# ─── Ön hazırlık ─────────────────────────────────────────────────────
# Bu kesim, kitapta bir önceki kesimde kurulan veriyi/modeli kullanır.
# Dosyanın tek başına çalışabilmesi için o hazırlık burada yinelenmiştir.
# Kaynak: bolum03/03_01_03_01_programatik-denetim-ile-eksik-veri-analizi.py
import numpy as np, pandas as pd
np.random.seed(42)
n = 200
df = pd.DataFrame({
    "yas":         np.where(np.random.rand(n) < 0.05, np.nan,
                            np.random.randint(18, 65, n).astype(float)),
    "gelir":       np.where(np.random.rand(n) < 0.12, np.nan,
                            np.random.normal(5000, 1500, n)),
    "egitim_yil":  np.where(np.random.rand(n) < 0.08, np.nan,
                            np.random.randint(8, 22, n).astype(float)),
    "kredi_skoru": np.where(np.random.rand(n) < 0.20, np.nan,
                            np.random.randint(300, 850, n).astype(float)),
    "sehir":       np.random.choice(["Ankara","Istanbul","Izmir",None],
                                     n, p=[0.35,0.35,0.20,0.10]),
})
# ─── Ön hazırlık sonu ────────────────────────────────────────────────

pattern_df = df.isnull().astype(int)
pattern_df["oruntu"] = pattern_df.apply(lambda r: "".join(r.astype(str)), axis=1)

pattern_count = pattern_df["oruntu"].value_counts().reset_index()
pattern_count.columns = ["Oruntu", "Satir Sayisi"]

cols = df.columns.tolist()
pattern_count["Eksik Sutunlar"] = pattern_count["Oruntu"].apply(
    lambda p: [c for c,b in zip(cols,p) if b=="1"] or ["Yok"]
)

print("Eksik veri oruntuleri (0=Mevcut, 1=Eksik):")
for _, row in pattern_count.head(10).iterrows():
    print(f"  {row['Oruntu']}  -> {row['Satir Sayisi']:3d} satir",
          f"| Eksik: {row['Eksik Sutunlar']}")

print(f"\nBenzersiz oruntu sayisi: {len(pattern_count)}")
print(f"Tam eksiksiz satir: %{df.dropna().shape[0]/len(df)*100:.1f}")


### 3.1.4.1. Yanlış Biçimlendirilmiş Veriler

`bolum03/03_01_04_01_yanlis-bicimlendirilmis-veriler.py`

_Kitap: Kod 3.5_


In [ ]:
import pandas as pd
import numpy as np

tarih_verileri = pd.Series([
    "2023-05-14",    # ISO 8601 - dogru
    "14/05/2023",    # Gun/Ay/Yil
    "May 14, 2023",  # Uzun bicim
    "14-05-23",      # Kisa yil
    "2023.05.14",    # Nokta ayracli
    "33/13/2023",    # Gecersiz gun/ay
    None,            # Bos deger
])

def tarihi_normalize_et(tarih_str):
    if pd.isnull(tarih_str):
        return pd.NaT
    bicimleri = ["%Y-%m-%d", "%d/%m/%Y", "%B %d, %Y",
                 "%d-%m-%y", "%Y.%m.%d", "%m/%d/%Y"]
    for bicim in bicimleri:
        try:
            return pd.to_datetime(tarih_str, format=bicim)
        except ValueError:
            continue
    return pd.NaT

normalize_tarihler = tarih_verileri.apply(tarihi_normalize_et)
sonuc = pd.DataFrame({
    "Ham Deger":      tarih_verileri,
    "Normalize":      normalize_tarihler,
    "Gecerli mi?":    normalize_tarihler.notna()
})
print(sonuc.to_string(index=False))


### 3.1.4.2. Tutarsız Veriler

`bolum03/03_01_04_02_tutarsiz-veriler.py`

_Kitap: Kod 3.6_


In [ ]:
import pandas as pd
from datetime import date

musteri_df = pd.DataFrame({
    "musteri_id":   [1,    2,          3,          4,          5],
    "dogum_tarihi": pd.to_datetime(["1990-03-15","2005-08-22",
                                    "1985-11-01","1978-06-30","2000-01-10"]),
    "beyan_yas":    [34,   18,         38,         72,         35],
})

bugun = pd.Timestamp(date.today())
musteri_df["hesap_yas"] = ((bugun - musteri_df["dogum_tarihi"]).dt.days / 365.25).astype(int)
musteri_df["yas_farki"]  = abs(musteri_df["beyan_yas"] - musteri_df["hesap_yas"])
musteri_df["tutarsiz"]   = musteri_df["yas_farki"] > 2

print("Yas tutarsizligi tespit edilen kayitlar:")
print(musteri_df[musteri_df["tutarsiz"]]
      [["musteri_id","dogum_tarihi","beyan_yas","hesap_yas","yas_farki"]]
      .to_string(index=False))


### 3.1.4.3. Tekrarlı Veriler (Duplicate Records)

`bolum03/03_01_04_03_tekrarli-veriler.py`

_Kitap: Kod 3.7_


In [ ]:
import pandas as pd
from difflib import SequenceMatcher

musteri_df = pd.DataFrame({
    "ad_soyad": ["Ahmet Yilmaz","Ahmet Yilmaz","Mehmet Kaya","mehmet kaya","Zeynep Sen"],
    "telefon":  ["5321112233","5321112233","5553334455","5553334455","5447778899"],
    "email":    ["a.y@m.com","a.y@m.com","m.k@m.com","mk@m.com","z.s@m.com"],
})

# 1) Tam tekrar
print(f"Tam tekrar satir sayisi: {musteri_df.duplicated(keep=False).sum()}")

# 2) Telefon numarasina gore grupla
print("\nAyni telefona sahip kayitlar:")
for tel, grp in musteri_df.groupby("telefon"):
    if len(grp) > 1:
        print(f"  Tel: {tel}")
        print(grp[["ad_soyad","email"]].to_string())

# 3) Fuzzy ad/soyad eslesme
adlar = musteri_df["ad_soyad"].tolist()
print("\nFuzzy eslesmeler (benzerlik >= 0.80):")
for i in range(len(adlar)):
    for j in range(i+1, len(adlar)):
        skor = SequenceMatcher(None,adlar[i].lower(),adlar[j].lower()).ratio()
        if skor >= 0.80:
            print(f"  [{i}]{adlar[i]} <-> [{j}]{adlar[j]} | Skor: {skor:.3f}")


### 3.1.4.4. Aykırı Değerler (Outliers)

`bolum03/03_01_04_04_aykiri-degerler.py`

_Kitap: Kod 3.8_


In [ ]:
import random
# ─── IQR yontemi ile aykiri deger tespiti ───────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(0)
degerler = np.concatenate([np.random.normal(50, 10, 200), [120, 130, -20, 160, 5]])
df_ay = pd.DataFrame({"deger": degerler})

Q1  = df_ay["deger"].quantile(0.25)
Q3  = df_ay["deger"].quantile(0.75)
IQR = Q3 - Q1
alt = Q1 - 1.5 * IQR
ust = Q3 + 1.5 * IQR

maske = (df_ay["deger"] < alt) | (df_ay["deger"] > ust)
print(f"Q1={Q1:.2f}, Q3={Q3:.2f}, IQR={IQR:.2f}")
print(f"Alt sinir: {alt:.2f}  |  Ust sinir: {ust:.2f}")
print(f"Aykiri deger sayisi: {maske.sum()}")
print(f"Aykiri degerler: {sorted(df_ay[maske]['deger'].values)}")

# Gorsellestirme
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
sns.boxplot(y=df_ay["deger"], ax=ax1, color="#4C9BE8")
ax1.axhline(alt, color="red", linestyle="--", label=f"Alt ({alt:.1f})")
ax1.axhline(ust, color="red", linestyle="--", label=f"Ust ({ust:.1f})")
ax1.set_title("Kutu Grafigi (IQR)", fontweight="bold")
ax1.legend()
sns.histplot(df_ay["deger"], bins=30, kde=True, ax=ax2, color="#4C9BE8")
ax2.axvline(alt, color="red", linestyle="--")
ax2.axvline(ust, color="red", linestyle="--")
ax2.set_title("Histogram + KDE", fontweight="bold")
plt.tight_layout()
plt.show()


### Z-Skoru ve Modified Z-Skoru

`bolum03/03_01_04_04_z-skoru-ve-modified-z-skoru.py`

_Kitap: Kod 3.9_


In [ ]:
import random
# ─── Z-Skoru ve Modified Z-Skoru ile aykiri deger tespiti ───────
import numpy as np
import pandas as pd
from scipy import stats

np.random.seed(42)
veri = np.concatenate([np.random.normal(50, 10, 300), [120, -30, 140, 200]])
df_z = pd.DataFrame({"deger": veri})

# Standart Z-Skoru
df_z["z_skoru"]  = stats.zscore(df_z["deger"])
df_z["z_aykiri"] = df_z["z_skoru"].abs() > 3

# Modified Z-Skoru
medyan = df_z["deger"].median()
mad    = np.median(np.abs(df_z["deger"] - medyan))
df_z["mz_skoru"]  = 0.6745 * (df_z["deger"] - medyan) / mad
df_z["mz_aykiri"] = df_z["mz_skoru"].abs() > 3.5

print(f"Z-Skoru    (|Z|>3.0)  aykiri sayisi: {df_z['z_aykiri'].sum()}")
print(f"Mod.Z-Sko. (|M|>3.5)  aykiri sayisi: {df_z['mz_aykiri'].sum()}")

print("\nZ-Skoru aykiri degerler:")
print(df_z[df_z["z_aykiri"]][["deger","z_skoru"]].sort_values("deger").to_string(index=False))


### 3.1.6.1. İstatistiksel Profilleme ile Bozuk Veri Tespiti

`bolum03/03_01_06_01_istatistiksel-profilleme-ile-bozuk-veri-tespiti.py`

_Kitap: Kod 3.10_


In [ ]:
import random
# ─── Kapsamli istatistiksel profil fonksiyonu ────────────────────
import pandas as pd
import numpy as np
from scipy import stats

def kapsamli_profil(df):
    profil_list = []
    for col in df.select_dtypes(include="number").columns:
        seri = df[col].dropna()
        if len(seri) == 0:
            continue
        Q1, Q3 = seri.quantile([0.25, 0.75])
        IQR = Q3 - Q1
        n_aykiri = ((seri < Q1-1.5*IQR) | (seri > Q3+1.5*IQR)).sum()
        _, p_norm = stats.shapiro(seri[:50]) if len(seri)>=3 else (None,None)
        profil_list.append({
            "Sutun":       col,
            "n":           int(len(seri)),
            "Eksik (%)":   round(df[col].isnull().mean()*100, 2),
            "Ort":         round(seri.mean(), 2),
            "Medyan":      round(seri.median(), 2),
            "Std":         round(seri.std(), 2),
            "Min":         round(seri.min(), 2),
            "Max":         round(seri.max(), 2),
            "Carpiklik":   round(seri.skew(), 3),
            "n_Aykiri":    int(n_aykiri),
            "Shapiro-p":   round(p_norm, 4) if p_norm else None,
        })
    return pd.DataFrame(profil_list)

# Ornek uygulama
np.random.seed(42)
df_test = pd.DataFrame({
    "gelir":  np.concatenate([np.random.normal(5000,1000,195),[50000,-100,np.nan,np.nan,np.nan]]),
    "yas":    np.concatenate([np.random.randint(18,65,198).astype(float),[999,-5]]),
    "puan":   np.random.randint(0,101,200).astype(float)
})

profil = kapsamli_profil(df_test)
print(profil.to_string(index=False))

print("\nOlasi sorunlu sutunlar:")
for _, r in profil.iterrows():
    sorunlar = []
    if r["Eksik (%)"] > 5:      sorunlar.append(f"Yuksek eksik (%{r['Eksik (%)']:.1f})")
    if abs(r["Carpiklik"]) > 1: sorunlar.append(f"Carpik ({r['Carpiklik']:.2f})")
    if r["n_Aykiri"] > 0:       sorunlar.append(f"{r['n_Aykiri']} aykiri")
    if sorunlar:
        print(f"  {r['Sutun']:12s}: {', '.join(sorunlar)}")


### 3.1.6.2. Kural Tabanlı Doğrulama

`bolum03/03_01_06_02_kural-tabanli-dogrulama.py`

_Kitap: Kod 3.11_


In [ ]:
import pandas as pd
import re

df_val = pd.DataFrame({
    "tckn":      ["12345678901","99999999999","123456789","12345678901","abc"],
    "email":     ["u@mail.com","eksik-at","u@d.org","","v@t.net"],
    "yas":       [25, -3, 200, 42, 18],
    "puan":      [85, 105, 90, -10, 77],
    "giris_yil": [2010, 2015, 2020, 2018, 2022],
    "cikis_yil": [2015, 2013, 2022, 2020, 2025],
})

hatalar = []

for idx, row in df_val.iterrows():
    if not str(row["tckn"]).isdigit() or len(str(row["tckn"])) != 11:
        hatalar.append((idx,"tckn","11 haneli sayisal olmali",row["tckn"]))

    if not re.match(r"^[^@\s]+@[^@\s]+\.[^@\s]+$", str(row["email"])):
        hatalar.append((idx,"email","Gecersiz e-posta bicimi",row["email"]))

    if not (0 <= row["yas"] <= 120):
        hatalar.append((idx,"yas","0-120 arasinda olmali",row["yas"]))

    if not (0 <= row["puan"] <= 100):
        hatalar.append((idx,"puan","0-100 arasinda olmali",row["puan"]))

    if row["cikis_yil"] < row["giris_yil"]:
        hatalar.append((idx,"cikis_yil","Giris yilindan once olamaz",row["cikis_yil"]))

hata_df = pd.DataFrame(hatalar, columns=["Satir","Sutun","Kural Ihlali","Deger"])
print(f"Toplam ihlal sayisi: {len(hata_df)}")
print(hata_df.to_string(index=False))


### 3.1.6.3. Görselleştirme Teknikleri

`bolum03/03_01_06_03_gorsellestirme-teknikleri.py`

_Kitap: Kod 3.12_


In [ ]:
import random
# ─── Cok panelli gorsel kalite raporu ───────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from scipy import stats

np.random.seed(7)
df_vis = pd.DataFrame({
    "gelir":   np.concatenate([np.random.lognormal(8.5,0.7,300),[500000,-1000]]),
    "yas":     np.concatenate([np.random.normal(40,12,300),[150,-5]]),
    "harcama": np.concatenate([np.random.normal(3000,800,300),[50000,-200]]),
})

renkler = ["#2196F3","#FF5722","#4CAF50"]
fig = plt.figure(figsize=(16, 12))
gs  = fig.add_gridspec(3, 3, hspace=0.45, wspace=0.35)

for i, col in enumerate(df_vis.columns):
    # Histogram + KDE
    ax_h = fig.add_subplot(gs[i, 0])
    sns.histplot(df_vis[col], bins=35, kde=True, ax=ax_h, color=renkler[i])
    ax_h.set_title(f"{col} - Histogram", fontsize=10)

    # Kutu grafigi
    ax_b = fig.add_subplot(gs[i, 1])
    sns.boxplot(y=df_vis[col], ax=ax_b, color=renkler[i])
    ax_b.set_title(f"{col} - Kutu Grafigi", fontsize=10)

    # Q-Q grafigi
    ax_q = fig.add_subplot(gs[i, 2])
    stats.probplot(df_vis[col].dropna(), dist="norm", plot=ax_q)
    ax_q.set_title(f"{col} - Q-Q Grafigi", fontsize=10)

plt.suptitle("Cok Degiskenli Veri Kalite Gorsel Raporu", fontsize=14, fontweight="bold")
plt.savefig("veri_kalite_raporu.png", dpi=150, bbox_inches="tight")
plt.show()


### 3.1.7.2. Tek Değişkenli İmputation

`bolum03/03_01_07_02_tek-degiskenli-imputation.py`

_Kitap: Kod 3.13_


In [ ]:
import random
# ─── SimpleImputer ile tek degiskenli doldurma ───────────────────
from sklearn.impute import SimpleImputer
import pandas as pd, numpy as np

np.random.seed(42)
df_imp = pd.DataFrame({
    "gelir": np.where(np.random.rand(100)<0.15, np.nan,
                      np.random.normal(5000,1000,100)),
    "yas":   np.where(np.random.rand(100)<0.10, np.nan,
                      np.random.randint(18,65,100).astype(float)),
    "sehir": np.where(np.random.rand(100)<0.08, np.nan,
                      np.random.choice(["Ankara","Istanbul","Izmir"],100)),
})

imp_ort = SimpleImputer(strategy="mean")
imp_med = SimpleImputer(strategy="median")

gelir_ort = pd.Series(imp_ort.fit_transform(df_imp[["gelir"]]).ravel())
gelir_med = pd.Series(imp_med.fit_transform(df_imp[["gelir"]]).ravel())

print("Gelir Std Sapma Karsilastirmasi:")
print(f"  Orijinal        : {df_imp['gelir'].std():.4f}")
print(f"  Ortalama ile    : {gelir_ort.std():.4f}")
print(f"  Medyan ile      : {gelir_med.std():.4f}")

# Kategorik: mod
imp_mod = SimpleImputer(strategy="most_frequent")
sehir_d = pd.Series(imp_mod.fit_transform(df_imp[["sehir"]]).ravel())
print("\nSehir - Mod ile Doldurma:")
print(sehir_d.value_counts())


### 3.1.7.3. KNN Imputation

`bolum03/03_01_07_03_knn-imputation.py`

_Kitap: Kod 3.14_


In [ ]:
import random
# ─── KNN Imputation ─────────────────────────────────────────────
from sklearn.impute import KNNImputer
import pandas as pd, numpy as np

np.random.seed(42)
df_knn = pd.DataFrame({
    "gelir":       np.where(np.random.rand(150)<0.15,np.nan,np.random.normal(5000,1000,150)),
    "yas":         np.where(np.random.rand(150)<0.10,np.nan,np.random.randint(18,65,150).astype(float)),
    "egitim_yil":  np.where(np.random.rand(150)<0.08,np.nan,np.random.randint(8,20,150).astype(float)),
    "kredi_skoru": np.where(np.random.rand(150)<0.12,np.nan,np.random.randint(300,850,150).astype(float)),
})

knn = KNNImputer(n_neighbors=5, weights="distance")
df_dolu = pd.DataFrame(knn.fit_transform(df_knn), columns=df_knn.columns)

print("Eksik deger - Once vs Sonra:")
print(pd.DataFrame({"Once": df_knn.isnull().sum(), "Sonra": df_dolu.isnull().sum()}))

print("\nGelir Istatistikleri:")
print(pd.DataFrame({"Orijinal": df_knn["gelir"].describe(),
                    "KNN":      df_dolu["gelir"].describe()}).round(2))


### 3.1.7.4. MICE (Multiple Imputation by Chained Equations)

`bolum03/03_01_07_04_mice.py`

_Kitap: Kod 3.15_


In [ ]:
import random
# ─── MICE: sklearn IterativeImputer ─────────────────────────────
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
import pandas as pd, numpy as np

np.random.seed(42)
df_m = pd.DataFrame({
    "gelir":      np.where(np.random.rand(200)<0.15,np.nan,np.random.normal(5000,1200,200)),
    "yas":        np.where(np.random.rand(200)<0.10,np.nan,np.random.randint(18,65,200).astype(float)),
    "egitim_yil": np.where(np.random.rand(200)<0.08,np.nan,np.random.randint(8,20,200).astype(float)),
})

mice = IterativeImputer(
    estimator=BayesianRidge(),
    max_iter=10,
    random_state=42,
    imputation_order="ascending"
)
df_m_dolu = pd.DataFrame(mice.fit_transform(df_m), columns=df_m.columns)

print("Gelir - MICE sonrasi istatistikler:")
print(pd.DataFrame({"Orijinal": df_m["gelir"].describe(),
                    "MICE":     df_m_dolu["gelir"].describe()}).round(2))
print(f"\nMICE sonrasi kalan eksik deger: {df_m_dolu.isnull().sum().sum()}")


### 3.1.8. Kapsamlı Örnek: Uçtan Uca Veri Kalite Pipeline'ı

`bolum03/03_01_08_kapsamli-ornek-uctan-uca-veri-kalite-pipeline-i.py`

_Kitap: Kod 3.16_


In [ ]:
import random
# ─── Uctan Uca Veri Kalite Pipeline ─────────────────────────────
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
import warnings
warnings.filterwarnings("ignore")

# 1) Kirli veri seti simulasyonu
np.random.seed(42)
n = 300
df_ham = pd.DataFrame({
    "musteri_id": range(1, n+1),
    "yas": np.concatenate([np.random.randint(18,65,n-10).astype(float),
                           [np.nan]*5, [-3,999,0,200,1500]]),
    "gelir": np.where(np.random.rand(n)<0.12, np.nan,
                      np.random.lognormal(8.5,0.6,n)),
    "harcama": np.where(np.random.rand(n)<0.08, np.nan,
                         np.random.normal(3000,800,n)),
    "sehir": np.where(np.random.rand(n)<0.07, np.nan,
                       np.random.choice(["Ankara","Istanbul","Izmir"],n)),
})

# 2) Kalite raporu fonksiyonu
def kalite_raporu(df, baslik):
    print(f"\n{'='*55}")
    print(f"  {baslik}  |  {len(df)} satir, {len(df.columns)} sutun")
    print(f"{'='*55}")
    eksik = df.isnull().sum()
    if eksik.any():
        for col, ne in eksik[eksik>0].items():
            print(f"  {col:15s}: {ne} eksik ({ne/len(df)*100:.1f}%)")
    else:
        print("  Tum degerler tam!")

kalite_raporu(df_ham, "HAM VERI")

# 3) IQR ile aykiri degerleri maskele
def iqr_maskele(df, col, k=1.5):
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    maske = (df[col] < Q1-k*IQR) | (df[col] > Q3+k*IQR)
    if maske.sum() > 0:
        print(f"  {col}: {maske.sum()} aykiri NaN yapildi")
    df.loc[maske, col] = np.nan
    return df

df_temiz = df_ham.copy()
print("\nAykiri Deger Temizleme:")
for col in ["yas","gelir","harcama"]:
    df_temiz = iqr_maskele(df_temiz, col)

# 4) Tekrarli kayitlari kaldir
n_once = len(df_temiz)
df_temiz = df_temiz.drop_duplicates(subset="musteri_id")
print(f"\nTekrarli kayit: {n_once-len(df_temiz)}")

# 5) Eksik degerleri doldur (KNN + mod)
sayisal = ["yas","gelir","harcama"]
kategorik = ["sehir"]

knn = KNNImputer(n_neighbors=5, weights="distance")
df_temiz[sayisal] = knn.fit_transform(df_temiz[sayisal])

for col in kategorik:
    mod = df_temiz[col].mode()[0]
    df_temiz[col] = df_temiz[col].fillna(mod)

# 6) Son kalite raporu
kalite_raporu(df_temiz, "TEMIZLENMIS VERI")

print(f"\nOzet:")
print(f"  Orijinal satir       : {len(df_ham)}")
print(f"  Temiz veri satir     : {len(df_temiz)}")
print(f"  Kalan eksik deger    : {df_temiz.isnull().sum().sum()}")


## 3.2. Veri Normalizasyonu ve Standartlaştırma


### Aykırı Değer Sorunu ve RobustScaler

`bolum03/03_02_01_01_aykiri-deger-sorunu-ve-robustscaler.py`

_Kitap: Kod 3.18_


In [ ]:
import random
# ─── MinMaxScaler vs RobustScaler Karşılaştırması ────────────────
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, RobustScaler

np.random.seed(7)
normal = np.random.normal(50, 10, 95)
aykiri = np.array([300, 350, 400, 450, 500])  # 5 aykırı değer
x = np.concatenate([normal, aykiri]).reshape(-1, 1)

mm  = MinMaxScaler().fit_transform(x)
rob = RobustScaler().fit_transform(x)

print("Karşılaştırma (aykırı değer: n=5, %5):")
print(f"  MinMax  — normal örnekler aralığı: [{mm[:95].min():.4f}, {mm[:95].max():.4f}]")
print(f"  Robust  — normal örnekler aralığı: [{rob[:95].min():.4f}, {rob[:95].max():.4f}]")
print(f"  MinMax  — aykırılar aralığı: [{mm[95:].min():.4f}, {mm[95:].max():.4f}]")
print(f"  Robust  — aykırılar aralığı: [{rob[95:].min():.4f}, {rob[95:].max():.4f}]")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, veri, baslik, renk in [
    (axes[0], x,   "Orijinal",  "#3498db"),
    (axes[1], mm,  "MinMaxScaler", "#e74c3c"),
    (axes[2], rob, "RobustScaler", "#2ecc71"),
]:
    ax.hist(veri, bins=30, color=renk, alpha=0.7)
    ax.set_title(baslik, fontweight="bold")
plt.suptitle("MinMax vs Robust: Aykırı Değer Etkisi", fontsize=12)
plt.tight_layout(); plt.show()


### 3.2.1.1. Min-Max Normalizasyonu (Lineer Ölçekleme)

`bolum03/03_02_01_01_min-max-normalizasyonu.py`

_Kitap: Kod 3.17_


In [ ]:
import random
# ─── Min-Max Normalizasyonu ──────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

np.random.seed(42)
df = pd.DataFrame({
    "yas":        np.random.randint(18, 65, 200).astype(float),
    "gelir":      np.random.normal(35000, 12000, 200),
    "kredi_skoru":np.random.randint(300, 850, 200).astype(float),
    "deneyim":    np.random.uniform(0, 40, 200),
})

# [0,1] ve [-1,1] aralıklarına normalizasyon
scaler01 = MinMaxScaler(feature_range=(0, 1))
scaler11 = MinMaxScaler(feature_range=(-1, 1))
df_n01 = pd.DataFrame(scaler01.fit_transform(df), columns=df.columns)
df_n11 = pd.DataFrame(scaler11.fit_transform(df), columns=df.columns)

# İstatistik karşılaştırması
print("=== [0,1] Normalizasyon Sonucu ===")
print(pd.DataFrame({
    "Orig Min": df.min().round(2),  "Orig Max": df.max().round(2),
    "Norm Min": df_n01.min().round(4), "Norm Max": df_n01.max().round(4),
}))

# Geri dönüşüm doğrulaması
df_geri = pd.DataFrame(scaler01.inverse_transform(df_n01), columns=df.columns)
print("\nGeri dönüşüm max mutlak hata:", (df - df_geri).abs().max().round(8).max())

# Aykırı değer etkisi
x_norm = np.array([10, 20, 30, 40, 500]).reshape(-1, 1)  # 500 aykırı
print("\nAykırı değer etkisi Min-Max:", MinMaxScaler().fit_transform(x_norm).ravel())


### 3.2.1.2. Ondalık Ölçeklendirme (Decimal Scaling)

`bolum03/03_02_01_02_ondalik-olceklendirme.py`

_Kitap: Kod 3.19_


In [ ]:
import numpy as np
import pandas as pd
import math

def decimal_scaling(x):
    """x' = x / 10^d,  d = ceil(log10(max|x|))"""
    x = np.asarray(x, dtype=float)
    max_abs = np.max(np.abs(x))
    if max_abs == 0: return x.copy(), 0
    d = math.ceil(math.log10(max_abs))
    return x / (10 ** d), d

# Test 1: Genel örnekler
ornekler = [
    ("Küçük tamsayılar", [12, 34, -7, 98, -23]),
    ("Yüzlük", [120, 345, -78, 987, -23]),
    ("Binlik (gelir)", [15000, 45000, 120000, 980000]),
    ("Ondalık", [0.05, 0.12, 0.87, -0.33]),
]

print("Veri Seti              d  Orijinal -> Olceklendirilmis"),
print("-"*75)
for isim, degerler in ornekler:
    arr = np.array(degerler, dtype=float)
    scaled, d = decimal_scaling(arr)
    print(f"{isim:<22} {d:>4}  {arr}  → {np.round(scaled,4)}")

# Test 2: DataFrame üzerinde
df = pd.DataFrame({"A":[10,50,90,30], "B":[1000,5500,9000,3200], "C":[0.01,0.05,0.09,0.03]})
df_sc = df.copy()
for col in df.columns:
    df_sc[col], d = decimal_scaling(df[col].values)
    print(f"Sütun {col}: d={d}, aralık=[{df_sc[col].min():.4f}, {df_sc[col].max():.4f}]")


### 3.2.1.4. L2 Normalizasyonu (Öklid Normu Tabanlı)

`bolum03/03_02_01_04_l2-normalizasyonu.py`

_Kitap: Kod 3.20_


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import normalize

# Kelime frekans matrisi (belge × kelime)
belgeler = np.array([
    [3, 0, 1, 0, 5, 2],   # Belge 1
    [0, 1, 0, 4, 0, 1],   # Belge 2
    [2, 2, 3, 0, 1, 0],   # Belge 3 (daha uzun)
    [0, 0, 0, 1, 2, 8],   # Belge 4 (kısa, bir kelime baskın)
], dtype=float)

# L1 normalizasyonu
b_l1 = normalize(belgeler, norm="l1")
print("L1 Normalize (her satır toplamı 1):")
print(np.round(b_l1, 4))
print("Satır L1 normları:", np.abs(b_l1).sum(axis=1))

# L2 normalizasyonu
b_l2 = normalize(belgeler, norm="l2")
print("\nL2 Normalize (her satır L2=1):")
print(np.round(b_l2, 4))
print("Satır L2 normları:", np.round(np.sqrt((b_l2**2).sum(axis=1)), 6))

# L2 normalize vektörler: nokta çarpımı = kosinus benzerliği
kosinus_sim = b_l2 @ b_l2.T
print("\nBelge Kosinus Benzerlik Matrisi:")
print(np.round(kosinus_sim, 3))

# Geometrik görselleştirme: 2B birim çember
v = np.array([[3,4],[1,7],[6,2],[5,5]], dtype=float)
v_l2 = normalize(v, norm="l2")
fig, (ax1,ax2) = plt.subplots(1,2,figsize=(10,5))
renkler = ["#e74c3c","#3498db","#2ecc71","#f39c12"]
for i,(vor,vn) in enumerate(zip(v,v_l2)):
    ax1.quiver(0,0,vor[0],vor[1],angles="xy",scale_units="xy",scale=1,color=renkler[i],label=f"v{i+1}")
    ax2.quiver(0,0,vn[0],vn[1],angles="xy",scale_units="xy",scale=1,color=renkler[i])
theta=np.linspace(0,2*np.pi,100)
ax2.plot(np.cos(theta),np.sin(theta),"k--",alpha=0.3)
ax1.set_xlim(-0.5, v[:,0].max()*1.15); ax1.set_ylim(-0.5, v[:,1].max()*1.15)
ax1.set_aspect("equal"); ax1.grid(alpha=0.3)
ax2.set_xlim(-1.2, 1.2); ax2.set_ylim(-1.2, 1.2); ax2.grid(alpha=0.3)
ax1.set_title("Orijinal Vektörler"); ax1.legend(fontsize=8)
ax2.set_title("L2 Normalize (Birim Çember)"); ax2.set_aspect("equal")
plt.tight_layout(); plt.show()


### 3.2.2.1. Z-Skoru Standartlaştırması

`bolum03/03_02_02_01_z-skoru-standartlastirmasi.py`

_Kitap: Kod 3.21_


In [ ]:
import random
# ─── Z-Skoru Standartlaştırması ──────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from scipy import stats

np.random.seed(42)
df = pd.DataFrame({
    "yas":         np.random.randint(18, 65, 300).astype(float),
    "gelir":       np.random.normal(50000, 15000, 300),
    "deneyim_yil": np.random.uniform(0, 40, 300),
    "kredi_skoru": np.random.randint(300, 850, 300).astype(float),
})

scaler = StandardScaler()
df_std = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)

# İstatistik doğrulama
print("=== Z-Skoru Standartlaştırması ===")
print(pd.DataFrame({
    "Orig Ort": df.mean().round(2),   "Orig Std": df.std().round(2),
    "Std Ort":  df_std.mean().round(6),"Std Std":  df_std.std().round(6),
}))

# Ampirik kural doğrulama
print("\nAmpirik Kural Doğrulama — gelir:")
z = df_std["gelir"]
for sinir, beklenen in [(1, 68.3), (2, 95.4), (3, 99.7)]:
    gercek = (z.abs() <= sinir).mean() * 100
    print(f"  |z| <= {sinir}: Beklenen ~%{beklenen:.1f}, Gerçek %{gercek:.1f}")

# Görselleştirme: histogram + standart normal eğrisi
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for i, col in enumerate(df.columns):
    axes[0,i].hist(df[col], bins=25, color="#3498db", alpha=0.7, density=True)
    axes[0,i].set_title(f"{col}\nOrijinal", fontsize=9)
    axes[1,i].hist(df_std[col], bins=25, color="#e74c3c", alpha=0.7, density=True)
    xr = np.linspace(-4,4,200)
    axes[1,i].plot(xr, stats.norm.pdf(xr), "k-", lw=2)
    axes[1,i].set_title(f"{col}\nZ-Skoru", fontsize=9)
plt.suptitle("Z-Skoru Standartlaştırması: Öncesi ve Sonrası", fontsize=12, fontweight="bold")
plt.tight_layout(); plt.show()


### 3.2.2.2. Robust Standartlaştırma (Medyan-IQR Tabanlı)

`bolum03/03_02_02_02_robust-standartlastirma.py`

_Kitap: Kod 3.22_


In [ ]:
import random
# ─── Z-Skoru vs RobustScaler Karşılaştırması ─────────────────────
import numpy as np
from sklearn.preprocessing import StandardScaler, RobustScaler
import matplotlib.pyplot as plt

np.random.seed(42)
n = 200
# %5 aykırı değer
normal = np.random.normal(50, 10, int(n*0.95))
aykiri = np.random.uniform(150, 300, int(n*0.05))
x = np.concatenate([normal, aykiri]).reshape(-1, 1)

std_sc = StandardScaler().fit_transform(x)
rob_sc = RobustScaler().fit_transform(x)

# print header omitted
print("-"*46)
for m, v1, v2 in [
    ("Ortalama",  std_sc.mean(),       rob_sc.mean()),
    ("Medyan",    np.median(std_sc),   np.median(rob_sc)),
    ("Std Sapma", std_sc.std(),        rob_sc.std()),
    ("IQR",       np.percentile(std_sc,75)-np.percentile(std_sc,25),
                  np.percentile(rob_sc,75)-np.percentile(rob_sc,25)),
]:
    print(f"{m:<22} {v1:>12.4f} {v2:>12.4f}")


### 3.2.2.3. MaxAbsScaler ve Seyrek Veri Ölçeklendirme

`bolum03/03_02_02_03_maxabsscaler-ve-seyrek-veri-olceklendirme.py`

_Kitap: Kod 3.23_


In [ ]:
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.preprocessing import MaxAbsScaler

# TF-IDF benzeri seyrek matris
X = np.array([[0,3,0,0,5],[2,0,0,4,0],[0,0,7,0,0],[1,2,0,0,3]], dtype=float)
X_sparse = csr_matrix(X)

scaler = MaxAbsScaler()
X_scaled = scaler.fit_transform(X_sparse)

print("MaxAbsScaler sonucu:")
print(X_scaled.toarray().round(4))
print(f"Korunan sıfır sayısı: {(X_scaled.toarray()==0).sum()} / {X.size}")
print("Ölçekleme faktörleri (her sütun için):", scaler.scale_)


### 3.2.3.2. Box-Cox ve Yeo-Johnson Dönüşümleri

`bolum03/03_02_03_02_box-cox-ve-yeo-johnson-donusumleri.py`

_Kitap: Kod 3.24_


In [ ]:
import random
# ─── Log, Box-Cox ve Yeo-Johnson Dönüşümleri ────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import PowerTransformer
from scipy import stats

np.random.seed(42)
gelir = np.random.lognormal(mean=10.5, sigma=0.8, size=500)

# Log dönüşümü
log_gelir = np.log(gelir)

# Box-Cox (yalnızca pozitif)
pt_bc = PowerTransformer(method="box-cox")
bc_gelir = pt_bc.fit_transform(gelir.reshape(-1,1)).ravel()

# Yeo-Johnson
pt_yj = PowerTransformer(method="yeo-johnson")
yj_gelir = pt_yj.fit_transform(gelir.reshape(-1,1)).ravel()

print(f"Box-Cox lambda   : {pt_bc.lambdas_[0]:.4f}")
print(f"Yeo-Johnson lambda: {pt_yj.lambdas_[0]:.4f}")

# Normallik testleri
print("\nNormallik Testleri (Shapiro-Wilk):")
idx = np.random.choice(500, 200, replace=False)
for isim, dizi in [("Orijinal", gelir[idx]), ("Log", log_gelir[idx]),
                    ("Box-Cox", bc_gelir[idx]), ("Yeo-Johnson", yj_gelir[idx])]:
    _, p = stats.shapiro(dizi)
    print(f"  {isim:<15}: p={p:.4f}  çarpıklık={stats.skew(dizi):.4f}")

# Görselleştirme
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
donusumler = [("Orijinal", gelir), ("log(x)", log_gelir),
              ("Box-Cox", bc_gelir), ("Yeo-Johnson", yj_gelir)]
for i, (isim, dizi) in enumerate(donusumler):
    axes[0,i].hist(dizi, bins=40, color="#3498db", alpha=0.7, density=True)
    axes[0,i].set_title(f"{isim}\nÇarp.:{stats.skew(dizi):.3f}", fontsize=9)
    stats.probplot(dizi, dist="norm", plot=axes[1,i])
    axes[1,i].set_title(f"Q-Q: {isim}", fontsize=9)
plt.suptitle("Güç Dönüşümleri: Normalliğe Yaklaştırma", fontsize=12, fontweight="bold")
plt.tight_layout(); plt.show()


### 3.2.4. Normalizasyon mu, Standartlaştırma mı? Seçim Rehberi

`bolum03/03_02_04_normalizasyon-mu-standartlastirma-mi-secim-rehbe.py`

_Kitap: Kod 3.25_


In [ ]:
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=1000, n_features=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# YANLIŞ: tüm veri üzerinde fit (data leakage!)
# scaler.fit_transform(X)  <- X test de içeriyor!

# DOĞRU: Pipeline kullanımı
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model",  LogisticRegression(max_iter=1000, random_state=42)),
])
cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring="accuracy")
print(f"Pipeline CV: %{cv_scores.mean()*100:.2f} ± %{cv_scores.std()*100:.2f}")

# ColumnTransformer: farklı sütunlara farklı dönüşüm
preprocessor = ColumnTransformer(transformers=[
    ("zscore", StandardScaler(),          list(range(5))),
    ("yeo",    PowerTransformer("yeo-johnson"), list(range(5,10))),
])
full_pipe = Pipeline([("prep", preprocessor), ("model", LogisticRegression(max_iter=1000))])
full_pipe.fit(X_train, y_train)
print(f"ColumnTransformer Test Doğruluğu: %{full_pipe.score(X_test,y_test)*100:.2f}")


### 3.2.6.1. Quantile Dönüşümü

`bolum03/03_02_06_01_quantile-donusumu.py`

_Kitap: Kod 3.26_


In [ ]:
import random
# ─── QuantileTransformer ─────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import QuantileTransformer
from scipy import stats

np.random.seed(42)
n = 1000
X = np.column_stack([
    np.random.lognormal(0, 1, n),   # Sağa çarpık
    np.random.exponential(2, n),    # Üstel
    stats.chi2.rvs(df=5, size=n),   # Ki-kare
])

qt_u = QuantileTransformer(output_distribution="uniform", n_quantiles=100, random_state=42)
qt_n = QuantileTransformer(output_distribution="normal",  n_quantiles=100, random_state=42)
X_u = qt_u.fit_transform(X)
X_n = qt_n.fit_transform(X)

isimler = ["Log-Normal", "Üstel", "Ki-kare"]
print("Normallik testi (Shapiro, n=200 alt örnek):")
# print header omitted
idx = np.random.choice(n, 200, replace=False)
for i, isim in enumerate(isimler):
    _, po = stats.shapiro(X[idx, i])
    _, pu = stats.shapiro(X_u[idx, i])
    _, pn = stats.shapiro(X_n[idx, i])
    print(f"{isim:<12} {po:>12.6f} {pu:>12.6f} {pn:>12.6f}")


### 3.2.6.2. Batch Normalizasyonu (Derin Öğrenme)

`bolum03/03_02_06_02_batch-normalizasyonu.py`

_Kitap: Kod 3.27_


In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

class MLP(nn.Module):
    def __init__(self, use_bn=False):
        super().__init__()
        layers = [nn.Linear(20, 64)]
        if use_bn: layers.append(nn.BatchNorm1d(64))
        layers += [nn.ReLU(), nn.Linear(64, 64)]
        if use_bn: layers.append(nn.BatchNorm1d(64))
        layers += [nn.ReLU(), nn.Linear(64, 1)]
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

torch.manual_seed(42)
n, d = 500, 20
# Farklı ölçeklerde özellikler (büyük ölçek farkı)
# 10**i, i=19'da int64 sinirini asiyor (OverflowError). Olcek farki ayni,
# ama ussu float olarak ve makul araliginda uretiyoruz.
olcekler = torch.tensor([10.0 ** (i % 6) for i in range(d)])
X = torch.randn(n, d) * olcekler
y = X[:, 0] * 0.5 + X[:, 1] * 0.3 + torch.randn(n) * 0.1

def train(model, epochs=150, lr=1e-3):
    opt, loss_fn = torch.optim.Adam(model.parameters(), lr=lr), nn.MSELoss()
    losses = []
    for _ in range(epochs):
        opt.zero_grad()
        loss = loss_fn(model(X).squeeze(), y)
        loss.backward(); opt.step()
        losses.append(loss.item())
    return losses

l_normal = train(MLP(use_bn=False))
l_bn     = train(MLP(use_bn=True))

plt.figure(figsize=(9, 4))
plt.plot(l_normal, label="MLP (BatchNorm yok)", color="#e74c3c", alpha=0.8)
plt.plot(l_bn,     label="MLP + BatchNorm",     color="#2ecc71", alpha=0.8)
plt.xlabel("Epoch"); plt.ylabel("MSE Loss")
plt.title("Batch Normalizasyonu: Eğitim Hızı Karşılaştırması")
plt.legend(); plt.yscale("log"); plt.tight_layout(); plt.show()


## 3.3. Anomali Tespiti


### 3.3.2.1. IQR Yontemi (Ceyrekler Arası Aclık)

`bolum03/03_03_02_01_iqr-yontemi.py`

_Kitap: Kod 3.28_


In [ ]:
import random
# IQR Yontemi ile Anomali Tespiti
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
n = 300
normal = np.random.normal(50, 10, n)
aykirilar = np.array([130, 140, -15, -25, 155, -30, 145])
veri = np.concatenate([normal, aykirilar])
df = pd.DataFrame({"deger": veri})

def iqr_anomali_tespit(dizi, k=1.5):
    Q1, Q3 = np.percentile(dizi, [25, 75])
    IQR = Q3 - Q1
    alt = Q1 - k * IQR
    ust = Q3 + k * IQR
    maske = (dizi < alt) | (dizi > ust)
    return maske, Q1, Q3, IQR, alt, ust

maske, Q1, Q3, IQR, alt, ust = iqr_anomali_tespit(df["deger"].values)

print("=== IQR Anomali Tespit Raporu ===")
print(f"  Q1={Q1:.3f}, Q3={Q3:.3f}, IQR={IQR:.3f}")
print(f"  Alt sinir (k=1.5): {alt:.3f}")
print(f"  Ust sinir (k=1.5): {ust:.3f}")
print(f"  Tespit edilen anomali sayisi: {maske.sum()}")
print(f"  Anomali degerleri: {sorted(df['deger'][maske].values)}")

# k=3.0 (asiri aykiri)
maske3, _, _, _, alt3, ust3 = iqr_anomali_tespit(df["deger"].values, k=3.0)
print(f"  k=3.0 => Sinirlar: [{alt3:.3f}, {ust3:.3f}] | Anomali: {maske3.sum()}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].boxplot(df["deger"], patch_artist=True, boxprops=dict(facecolor="#AED6F1"),
                medianprops=dict(color="red", lw=2))
axes[0].set_title("Kutu Grafigi ile IQR", fontweight="bold")
normal_pts = df["deger"][~maske]
aykiri_pts = df["deger"][maske]
axes[1].scatter(range(len(normal_pts)), normal_pts.values, c="#3498db", alpha=0.5, s=20, label="Normal")
axes[1].scatter(range(len(aykiri_pts)), aykiri_pts.values, c="#e74c3c", s=80, marker="x", lw=2, label="Anomali")
axes[1].axhline(ust, color="orange", linestyle="--", label=f"Ust {ust:.1f}")
axes[1].axhline(alt, color="green", linestyle="--", label=f"Alt {alt:.1f}")
axes[1].legend(); axes[1].set_title("Anomali Noktalari", fontweight="bold")
plt.tight_layout(); plt.show()


### 3.3.2.2. Z-Skoru ve Modified Z-Skoru

`bolum03/03_03_02_02_z-skoru-ve-modified-z-skoru.py`

_Kitap: Kod 3.29_


In [ ]:
import random
# Z-Skoru ve Modified Z-Skoru Karsilastirmasi
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
n = 500
normal = np.random.normal(50, 8, int(n*0.95))
aykiri = np.array([110.0, 115.0, 120.0, -15.0, -20.0, -25.0,
                    110.0, 115.0, 120.0, -15.0, -20.0, -25.0,
                    110.0, 115.0, 120.0, -15.0, -20.0, -25.0,
                    110.0, 115.0, 120.0, -15.0, -20.0, -25.0,
                    110.0, 115.0])
x = np.concatenate([normal, aykiri])

# Standart Z-Skoru
mu, sigma = x.mean(), x.std()
z = (x - mu) / sigma
maske_z = np.abs(z) > 3.0

# Modified Z-Skoru
medyan = np.median(x)
mad = np.median(np.abs(x - medyan))
M = 0.6745 * (x - medyan) / mad
maske_mz = np.abs(M) > 3.5

print("Z-Skoru (|z|>3.0)    : {} anomali".format(maske_z.sum()))
print("Mod Z-Skoru (|M|>3.5): {} anomali".format(maske_mz.sum()))

# Maskeleme etkisi
x2 = np.concatenate([normal, np.array([200.0]*15)])
z2 = (x2 - x2.mean()) / x2.std()
M2 = 0.6745*(x2 - np.median(x2)) / np.median(np.abs(x2 - np.median(x2)))
print("Maskeleme (15 buyuk aykiri):")
print("  Z-Skoru anomali: {} (maskeleme riski!)".format((np.abs(z2)>3).sum()))
print("  Mod Z-Skoru    : {} (maskelemeye dayanikli)".format((np.abs(M2)>3.5).sum()))


### 3.3.2.3. Gaussian Mixture Model (GMM) ile Anomali Tespiti

`bolum03/03_03_02_03_gaussian-mixture-model-ile-anomali-tespiti.py`

_Kitap: Kod 3.30_


In [ ]:
import random
# GMM ile Anomali Tespiti
import numpy as np
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture

np.random.seed(42)
X_k1 = np.random.multivariate_normal([2, 2],  [[1,0],[0,1]],   150)
X_k2 = np.random.multivariate_normal([8, 8],  [[1.5,0],[0,1.5]],100)
X_ay = np.random.uniform(-5, 13, (15, 2))
X = np.vstack([X_k1, X_k2, X_ay])

gmm = GaussianMixture(n_components=2, covariance_type="full", random_state=42)
gmm.fit(X)
log_prob = gmm.score_samples(X)
esik = np.percentile(log_prob, 5)
anomali_maske = log_prob < esik

print("GMM esik (5. yuzdelik): {:.4f}".format(esik))
print("Anomali sayisi: {}".format(anomali_maske.sum()))

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(X[~anomali_maske,0], X[~anomali_maske,1], c="#3498db", s=20, alpha=0.6, label="Normal")
ax.scatter(X[anomali_maske,0],  X[anomali_maske,1],  c="#e74c3c", s=80, marker="X", label="Anomali")
ax.set_title("GMM Anomali Tespiti", fontweight="bold"); ax.legend(); plt.show()


### 3.3.3.1. DBSCAN (Density-Based Spatial Clustering of Applications with Noise)

`bolum03/03_03_03_01_dbscan.py`

_Kitap: Kod 3.31_


In [ ]:
import random
# DBSCAN ile Anomali Tespiti
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

np.random.seed(42)
theta = np.linspace(0, 2*np.pi, 200)
r1 = 3 + np.random.normal(0, 0.3, 200)
ring1 = np.c_[r1*np.cos(theta), r1*np.sin(theta)]
blob1 = np.random.multivariate_normal([0,0],[[0.5,0],[0,0.5]],150)
blob2 = np.random.multivariate_normal([7,3],[[0.8,0],[0,0.8]],100)
anomali = np.array([[6,0],[8,-2],[-4,5],[-5,-4],[9,6],[-6,3]])
X = np.vstack([ring1, blob1, blob2, anomali])
X_s = StandardScaler().fit_transform(X)

# k-Mesafe grafigi ile epsilon secimi
knn = NearestNeighbors(n_neighbors=5).fit(X_s)
mesafeler, _ = knn.kneighbors(X_s)
kinci_dist = np.sort(mesafeler[:,4])[::-1]

dbscan = DBSCAN(eps=0.3, min_samples=5)
etiketler = dbscan.fit_predict(X_s)
n_kume = len(set(etiketler)) - (1 if -1 in etiketler else 0)
n_anom = (etiketler == -1).sum()
print("Kume sayisi : {}".format(n_kume))
print("Anomali     : {}".format(n_anom))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(kinci_dist, color="#3498db")
ax1.axhline(0.3, color="red", linestyle="--", label="eps=0.3")
ax1.set_title("k-Mesafe Grafigi (eps secimi)"); ax1.legend()
for lbl in set(etiketler):
    pts = X[etiketler==lbl]
    c = "#e74c3c" if lbl==-1 else plt.cm.tab10(lbl%10)
    ax2.scatter(pts[:,0],pts[:,1],c=[c],s=15 if lbl!=-1 else 80,
                marker="o" if lbl!=-1 else "X",
                label="Anomali" if lbl==-1 else "Kume {}".format(lbl))
ax2.set_title("DBSCAN (eps=0.3, MinPts=5)"); ax2.legend(fontsize=8)
plt.tight_layout(); plt.show()


### 3.3.3.2. LOF (Local Outlier Factor)

`bolum03/03_03_03_02_lof.py`

_Kitap: Kod 3.32_


In [ ]:
import random
# LOF (Local Outlier Factor) ile Anomali Tespiti
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import LocalOutlierFactor

np.random.seed(42)
kume1 = np.random.multivariate_normal([0,0],[[0.3,0],[0,0.3]],100)
kume2 = np.random.multivariate_normal([6,0],[[1.5,0],[0,1.5]], 50)
kume3 = np.random.multivariate_normal([3,5],[[0.5,0],[0,0.5]], 60)
anomali = np.array([[10,8],[-4,5],[3,-4],[8,-3],[-3,-5],[12,2]])
X = np.vstack([kume1, kume2, kume3, anomali])

lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05)
tahmin = lof.fit_predict(X)
lof_skor = -lof.negative_outlier_factor_

print("Anomali sayisi: {}".format((tahmin==-1).sum()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
normal_pts = X[tahmin==1]; anomali_pts = X[tahmin==-1]
axes[0].scatter(normal_pts[:,0], normal_pts[:,1], c="#3498db", s=20, alpha=0.7, label="Normal")
axes[0].scatter(anomali_pts[:,0],anomali_pts[:,1],c="#e74c3c", s=120, marker="X", label="Anomali")
axes[0].set_title("LOF Anomali Tespiti (k=20)"); axes[0].legend()
sc = axes[1].scatter(X[:,0], X[:,1], c=lof_skor, cmap="RdYlBu_r", s=lof_skor*5, alpha=0.7, vmin=1, vmax=5)
plt.colorbar(sc, ax=axes[1], label="LOF Skoru")
axes[1].set_title("LOF Skor Yogunlugu")
plt.tight_layout(); plt.show()


### 3.3.4.1. Isolation Forest

`bolum03/03_03_04_01_isolation-forest.py`

_Kitap: Kod 3.33_


In [ ]:
import random
# Isolation Forest ile Anomali Tespiti
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score

np.random.seed(42)
n_norm, n_anom = 1000, 50
X_norm = np.random.multivariate_normal([5,5,5,5], np.eye(4)*1.5, n_norm)
X_anom = np.random.uniform(low=-5, high=15, size=(n_anom, 4))
X = np.vstack([X_norm, X_anom])
y = np.array([1]*n_norm + [-1]*n_anom)
X_s = StandardScaler().fit_transform(X)

isof = IsolationForest(n_estimators=100, contamination=0.05, max_samples=256, random_state=42)
isof.fit(X_s)
tahmin = isof.predict(X_s)
skorlar = isof.decision_function(X_s)

y_bin = (y==-1).astype(int)
p_bin = (tahmin==-1).astype(int)
print(classification_report(y_bin, p_bin, target_names=["Normal","Anomali"]))
print("ROC-AUC: {:.4f}".format(roc_auc_score(y_bin, -skorlar)))

# contamination etkisi
for cont in [0.01, 0.03, 0.05, 0.08, 0.10]:
    prd = IsolationForest(contamination=cont, random_state=42).fit_predict(X_s)
    tp = ((prd==-1)&(y==-1)).sum()
    fp = ((prd==-1)&(y==1)).sum()
    fn = ((prd==1)&(y==-1)).sum()
    print("  cont={:.2f}: TP={}, FP={}, FN={}".format(cont,tp,fp,fn))


### 3.3.4.2. One-Class SVM (OC-SVM)

`bolum03/03_03_04_02_one-class-svm.py`

_Kitap: Kod 3.34_


In [ ]:
import random
# One-Class SVM ile Anomali Tespiti
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

np.random.seed(42)
X_eg = np.random.multivariate_normal([3,3],[[1.5,0.5],[0.5,1.5]],200)
X_te_n = np.random.multivariate_normal([3,3],[[1.5,0.5],[0.5,1.5]],100)
X_te_a = np.random.uniform(-3, 9, (20, 2))
X_te = np.vstack([X_te_n, X_te_a])
y_te = np.array([1]*100+[-1]*20)

sc = StandardScaler().fit(X_eg)
X_eg_s = sc.transform(X_eg)
X_te_s = sc.transform(X_te)

ocsvm = OneClassSVM(kernel="rbf", nu=0.1, gamma="scale")
ocsvm.fit(X_eg_s)
tahmin = ocsvm.predict(X_te_s)
print(classification_report(y_te, tahmin, target_names=["Anomali","Normal"]))

# Izgara sinirlarini veriden turet (sabit -4..7 araligi uc noktalari disarida birakiyordu)
_tum = np.vstack([X_eg_s, X_te_s])
_pad = 0.8
xx, yy = np.meshgrid(
    np.linspace(_tum[:,0].min()-_pad, _tum[:,0].max()+_pad, 200),
    np.linspace(_tum[:,1].min()-_pad, _tum[:,1].max()+_pad, 200))
Z = ocsvm.decision_function(np.c_[xx.ravel(),yy.ravel()]).reshape(xx.shape)
fig, ax = plt.subplots(figsize=(8,6))
ax.contourf(xx, yy, Z, levels=15, cmap="RdBu_r", alpha=0.5)
ax.contour(xx, yy, Z, levels=[0], colors="black", linewidths=2)
ax.scatter(X_eg_s[:,0], X_eg_s[:,1], c="#95a5a6", s=15, alpha=0.4, label="Egitim")
ax.scatter(X_te_s[tahmin==1,0], X_te_s[tahmin==1,1], c="#3498db", s=40, label="Normal(test)")
ax.scatter(X_te_s[tahmin==-1,0],X_te_s[tahmin==-1,1],c="#e74c3c",s=80,marker="X",label="Anomali")
ax.set_title("One-Class SVM: Karar Siniri (RBF)"); ax.legend(); plt.show()


### 3.3.5.1. Autoencoder Tabanlı Anomali Tespiti

`bolum03/03_03_05_01_autoencoder-tabanli-anomali-tespiti.py`

_Kitap: Kod 3.35_


In [ ]:
import random
# Autoencoder ile Zaman Serisi Anomali Tespiti
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

torch.manual_seed(42); np.random.seed(42)
t = np.linspace(0, 100, 2000)
ts = np.sin(2*np.pi*t/10) + np.sin(2*np.pi*t/3) + np.random.normal(0,0.2,len(t))
anom_idx = np.random.choice(range(500,1800),10,replace=False)
ts_anom = ts.copy(); ts_anom[anom_idx] += np.random.uniform(3,5,10)

window = 30
X_w = np.array([ts[i:i+window] for i in range(len(ts)-window)], dtype=np.float32)
X_t = torch.FloatTensor(X_w)

class AE(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(30,16),nn.ReLU(),nn.Linear(16,6),nn.ReLU())
        self.dec = nn.Sequential(nn.Linear(6,16),nn.ReLU(),nn.Linear(16,30))
    def forward(self,x): return self.dec(self.enc(x))

model = AE()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
ldr = DataLoader(TensorDataset(X_t), batch_size=64, shuffle=True)
model.train()
for ep in range(50):
    tot=0
    for (b,) in ldr:
        r=model(b); l=nn.MSELoss()(r,b)
        opt.zero_grad(); l.backward(); opt.step(); tot+=l.item()
    if (ep+1)%10==0: print("Epoch {}: {:.6f}".format(ep+1,tot/len(ldr)))

model.eval()
with torch.no_grad():
    yeniden = model(X_t)
    hatalar = ((X_t-yeniden)**2).mean(dim=1).numpy()
esik = np.percentile(hatalar, 95)
print("Esik (95.yuzdelik): {:.6f}".format(esik))
print("Anomali sayisi: {}".format((hatalar>esik).sum()))


### 3.3.6.1. Temel Degerlendirme Metrikleri

`bolum03/03_03_06_01_temel-degerlendirme-metrikleri.py`

_Kitap: Kod 3.36_


In [ ]:
import random
# Kapsamlı Model Degerlendirme
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score,
                             roc_curve, precision_recall_curve)
from sklearn.model_selection import train_test_split
import warnings; warnings.filterwarnings("ignore")

np.random.seed(42)
n_norm, n_anom = 950, 50
X_n = np.random.multivariate_normal([5,5],[[2,0.5],[0.5,2]],n_norm)
X_a = np.random.uniform(-3,13,(n_anom,2))
X = np.vstack([X_n, X_a])
y = np.array([0]*n_norm+[1]*n_anom)
X_tr,X_te,y_tr,y_te = train_test_split(X,y,test_size=0.3,stratify=y,random_state=42)
sc = StandardScaler().fit(X_tr)
X_tr_s = sc.transform(X_tr); X_te_s = sc.transform(X_te)

modeller = {
    "Isolation Forest": IsolationForest(contamination=0.05, random_state=42),
    "LOF (k=20)": LocalOutlierFactor(n_neighbors=20, contamination=0.05, novelty=True),
    "One-Class SVM": OneClassSVM(nu=0.05, kernel="rbf", gamma="scale"),
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
renkler = {"Isolation Forest":"#e74c3c","LOF (k=20)":"#3498db","One-Class SVM":"#2ecc71"}

for isim, model in modeller.items():
    model.fit(X_tr_s)
    pred = (model.predict(X_te_s)==-1).astype(int)
    skor = -model.decision_function(X_te_s)
    pr = precision_score(y_te,pred,zero_division=0)
    rc = recall_score(y_te,pred,zero_division=0)
    f1 = f1_score(y_te,pred,zero_division=0)
    auc = roc_auc_score(y_te,skor)
    print("{}: P={:.3f} R={:.3f} F1={:.3f} AUC={:.3f}".format(isim,pr,rc,f1,auc))
    fpr,tpr,_ = roc_curve(y_te,skor)
    p,r,_ = precision_recall_curve(y_te,skor)
    ax1.plot(fpr,tpr,color=renkler[isim],lw=2,label="{} ({:.3f})".format(isim,auc))
    ax2.plot(r,p,color=renkler[isim],lw=2,label="{} ({:.3f})".format(isim,average_precision_score(y_te,skor)))

ax1.plot([0,1],[0,1],"k--"); ax1.set_title("ROC Egrisi"); ax1.legend(fontsize=8)
ax2.set_title("Precision-Recall Egrisi"); ax2.legend(fontsize=8)
plt.tight_layout(); plt.show()


### 3.3.7. Kapsamlı Ornek: Uctan Uca Anomali Tespit Pipeline'ı

`bolum03/03_03_07_kapsamli-ornek-uctan-uca-anomali-tespit-pipeline.py`

_Kitap: Kod 3.37_


In [ ]:
import random
# Uctan Uca Anomali Tespit Pipeline'ı — Banka Islemi Senaryosu
import numpy as np, pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

np.random.seed(42)
n = 2000
df = pd.DataFrame({
    "tutar":  np.concatenate([np.random.lognormal(5,1,int(n*0.97)),np.random.uniform(5000,20000,int(n*0.03))]),
    "saat":   np.concatenate([np.random.randint(8,22,int(n*0.97)),np.random.randint(0,6,int(n*0.03))]).astype(float),
    "gun_is": np.concatenate([np.random.poisson(3,int(n*0.97)),np.random.poisson(25,int(n*0.03))]).astype(float),
    "yabanci":np.concatenate([np.zeros(int(n*0.97)),np.ones(int(n*0.03))]),
    "etiket": np.concatenate([np.zeros(int(n*0.97)),np.ones(int(n*0.03))]),
})
df["log_tutar"] = np.log1p(df["tutar"])
df["gece"]      = ((df["saat"]<6)|(df["saat"]>23)).astype(int)

ozl = ["log_tutar","saat","gun_is","yabanci","gece"]
X = StandardScaler().fit_transform(df[ozl].values)
y = df["etiket"].values

# Ensemble skor
skor_if  = -IsolationForest(contamination=0.03,random_state=42).fit(X).decision_function(X)
skor_lof = -LocalOutlierFactor(n_neighbors=20).fit(X).negative_outlier_factor_
skor_if_n  = (skor_if -skor_if.min())/(skor_if.max()-skor_if.min())
skor_lof_n = (skor_lof-skor_lof.min())/(skor_lof.max()-skor_lof.min())
skor_ens   = (skor_if_n + skor_lof_n) / 2

# Esik optimizasyonu
esikler = np.linspace(0.3, 0.9, 100)
en_iyi_f1, en_iyi_esik = 0, 0.5
for e in esikler:
    pred = (skor_ens > e).astype(int)
    f1 = f1_score(y.astype(int), pred, zero_division=0)
    if f1 > en_iyi_f1: en_iyi_f1, en_iyi_esik = f1, e

tahmin = (skor_ens > en_iyi_esik).astype(int)
print("Optimal esik: {:.3f} | F1: {:.4f}".format(en_iyi_esik, en_iyi_f1))
print(classification_report(y.astype(int), tahmin, target_names=["Normal","Anomali"]))


### 3.3.8. Zaman Serisi Anomali Tespiti

`bolum03/03_03_08_zaman-serisi-anomali-tespiti.py`

_Kitap: Kod 3.38_


In [ ]:
import random
# Zaman Serisi Anomali Tespiti: Hareketli Z-Skoru
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest

np.random.seed(42)
T = 730
tarihler = pd.date_range("2022-01-01", periods=T, freq="D")
trend   = np.linspace(100, 150, T)
mevsim  = 20 * np.sin(2*np.pi*np.arange(T)/365)
haftalik= 10 * np.sin(2*np.pi*np.arange(T)/7)
ts = trend + mevsim + haftalik + np.random.normal(0,5,T)
anom_idx = [50,100,200,350,500,600,680]
for idx in anom_idx: ts[idx:idx+3] += np.random.choice([-1,1])*np.random.uniform(30,60)

df_ts = pd.DataFrame({"deger":ts}, index=tarihler)
pencere = 30
df_ts["ort"] = df_ts["deger"].rolling(pencere,center=True).mean()
df_ts["std"] = df_ts["deger"].rolling(pencere,center=True).std()
df_ts["z_t"] = (df_ts["deger"]-df_ts["ort"])/(df_ts["std"]+1e-9)
df_ts["anom_z"] = df_ts["z_t"].abs()>3.0
print("Hareketli Z-Skoru anomali sayisi: {}".format(df_ts["anom_z"].sum()))

fig, (ax1,ax2) = plt.subplots(2,1,figsize=(16,8),sharex=True)
ax1.plot(df_ts.index, df_ts["deger"], color="#3498db", linewidth=0.8, alpha=0.8)
ax1.scatter(df_ts.index[df_ts["anom_z"]], df_ts["deger"][df_ts["anom_z"]],
            c="#e74c3c", s=60, zorder=5, label="Anomali")
ax1.set_title("Hareketli Z-Skoru Anomali Tespiti (pencere=30)"); ax1.legend()
ax2.plot(df_ts.index, df_ts["z_t"], color="#2ecc71", linewidth=0.8)
ax2.axhline(3.0, color="red", linestyle="--"); ax2.axhline(-3.0, color="red", linestyle="--")
ax2.set_title("Hareketli Z-Skoru (|z|>3 = anomali)")
plt.tight_layout(); plt.show()
